# mT5-Large BM25-Focused Context Training

This notebook trains a new focused-context model for Arabic Palestinian narrative QA.

Experiment idea:

1. Load `data_qa.csv`.
2. Recreate the same leakage-aware train/validation/test split.
3. Build BM25-focused context inside each row's own `context_text`, using the question only.
4. Fine-tune `google/mt5-large` on the focused-context training split.
5. Predict on the focused-context test split.
6. Recompute honest metrics with Arabic-normalized ROUGE-L, ROUGE-S, and Token-F1.

BM25 is used only to select relevant sentences from the already available gold context.

## 1. Install dependency

In [1]:
# try:
#     from rank_bm25 import BM25Okapi
# except ImportError:
#     import sys
#     !{sys.executable} -m pip install rank-bm25
#     from rank_bm25 import BM25Okapi

## 2. Imports and configuration

In [ ]:
import os
import re
import gc
import math
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import sacrebleu

from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from bert_score import score as bertScore
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi

seed = 3407

baseModelName = "google/mt5-large"

# modelStartPath = "./outputs/google/mt5-large-full-context-validated"
modelStartPath = baseModelName

modelOutputDir = Path("./outputs/google/mt5-large-bm25-full-tuning")
resultsDir = Path("./outputs/full_context_bm25_full-tuning_results")
resultsDir.mkdir(parents=True, exist_ok=True)

dataPath = Path("data_qa.csv")

trainRatio = 0.80
validationRatio = 0.10

maxInputLength = 1024
maxTargetLength = 256

trainBatchSize = 8
evalBatchSize = 8
gradientAccumulationSteps = 8
numTrainEpochs = 5
learningRate = 5e-5
weightDecay = 0.01
warmupRatio = 0.05

freezeEncoder = False

generationNumBeams = 1
generationMaxNewTokens = maxTargetLength

focusedTopK = 5
focusedWindowSize = 1
focusedMaxWords = 320

forceRegeneratePredictions = True

semanticMetricModelName = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(seed)
np.random.seed(seed)

print("device:", device)
print("modelStartPath:", modelStartPath)
print("modelOutputDir:", modelOutputDir)
print("resultsDir:", resultsDir)

ModuleNotFoundError: No module named 'sacrebleu'

## 3. Text cleaning, Arabic normalization, and metrics

In [ ]:
def normalizeText(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def cleanReference(text):
    text = str(text)
    text = re.sub(r"\s*هذا الجواب مستند إلى المصدر التالي:\s*\S+\s*$", "", text).strip()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def cleanGeneratedText(text):
    text = normalizeText(text)
    text = re.sub(r"<extra_id_\d+>", " ", text)
    text = text.replace("<pad>", " ").replace("</s>", " ").replace("<s>", " ")
    text = re.sub(r"^(الإجابة|الجواب)\s*[:：]\s*", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalizeArabicText(text):
    text = str(text)
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    text = re.sub(r"[إأآٱا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)

    text = re.sub(r"گ", "ك", text)
    text = re.sub(r"چ", "ج", text)
    text = re.sub(r"پ", "ب", text)
    text = re.sub(r"ڤ", "ف", text)

    text = re.sub(r"ـ", "", text)

    text = re.sub(r"[“”\"'`´،؛؟!.,:()\[\]{}<>«»]", " ", text)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def tokenizeArabic(text):
    return normalizeArabicText(text).split()

def lcsLength(tokensA, tokensB):
    rows = len(tokensA) + 1
    cols = len(tokensB) + 1
    dp = [[0] * cols for _ in range(rows)]

    for i in range(1, rows):
        for j in range(1, cols):
            if tokensA[i - 1] == tokensB[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    return dp[-1][-1]

def rougeLArabic(prediction, reference):
    predTokens = tokenizeArabic(prediction)
    refTokens = tokenizeArabic(reference)

    if len(predTokens) == 0 or len(refTokens) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    lcs = lcsLength(predTokens, refTokens)
    precision = lcs / len(predTokens)
    recall = lcs / len(refTokens)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}

def getSkipBigrams(tokens, maxSkip=4):
    bigrams = set()

    for i in range(len(tokens)):
        for skip in range(1, maxSkip + 2):
            j = i + skip

            if j < len(tokens):
                bigrams.add((tokens[i], tokens[j]))

    return bigrams

def rougeSArabic(prediction, reference, maxSkip=4):
    predTokens = tokenizeArabic(prediction)
    refTokens = tokenizeArabic(reference)

    predBigrams = getSkipBigrams(predTokens, maxSkip=maxSkip)
    refBigrams = getSkipBigrams(refTokens, maxSkip=maxSkip)

    if not predBigrams or not refBigrams:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    overlap = len(predBigrams & refBigrams)
    precision = overlap / len(predBigrams)
    recall = overlap / len(refBigrams)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}

def tokenF1Arabic(prediction, reference):
    predTokens = tokenizeArabic(prediction)
    refTokens = tokenizeArabic(reference)

    if len(predTokens) == 0 or len(refTokens) == 0:
        return 0.0

    overlap = sum((Counter(predTokens) & Counter(refTokens)).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(predTokens)
    recall = overlap / len(refTokens)

    return 2 * precision * recall / (precision + recall)

def exactMatchArabic(prediction, reference):
    return int(normalizeArabicText(prediction) == normalizeArabicText(reference))

def containsNoAnswer(text):
    normalized = normalizeArabicText(text)
    return int("لا تتوفر معلومات كافيه" in normalized or "لا توجد معلومات كافيه" in normalized)

def computeMetricsHonest(predictions, references):
    predictions = [cleanGeneratedText(value) for value in predictions]
    references = [cleanReference(value) for value in references]

    bleu = sacrebleu.corpus_bleu(predictions, [references]).score
    chrf = sacrebleu.corpus_chrf(predictions, [references]).score

    bertPrecision, bertRecall, bertF1 = bertScore(
        predictions,
        references,
        lang="ar",
        verbose=False,
        rescale_with_baseline=False,
        device=device
    )

    semanticMetricModel = SentenceTransformer(semanticMetricModelName, device=device)

    predEmbeddings = semanticMetricModel.encode(
        predictions,
        convert_to_tensor=True,
        show_progress_bar=True
    )

    refEmbeddings = semanticMetricModel.encode(
        references,
        convert_to_tensor=True,
        show_progress_bar=True
    )

    semanticSimilarities = util.cos_sim(predEmbeddings, refEmbeddings).diagonal()

    rougeLResults = [
        rougeLArabic(prediction, reference)
        for prediction, reference in zip(predictions, references)
    ]

    rougeSResults = [
        rougeSArabic(prediction, reference)
        for prediction, reference in zip(predictions, references)
    ]

    tokenF1Scores = [
        tokenF1Arabic(prediction, reference)
        for prediction, reference in zip(predictions, references)
    ]

    exactMatchScores = [
        exactMatchArabic(prediction, reference)
        for prediction, reference in zip(predictions, references)
    ]

    noAnswerScores = [
        containsNoAnswer(prediction)
        for prediction in predictions
    ]

    predictionLengths = [len(tokenizeArabic(prediction)) for prediction in predictions]
    referenceLengths = [len(tokenizeArabic(reference)) for reference in references]

    metrics = {
        "BLEU": bleu,
        "chrF": chrf,
        "BERTScore_Precision": float(bertPrecision.mean()),
        "BERTScore_Recall": float(bertRecall.mean()),
        "BERTScore_F1": float(bertF1.mean()),
        "Semantic_Textual_Similarity": float(semanticSimilarities.mean()),

        "Arabic_ROUGE_L_Precision": float(np.mean([item["precision"] for item in rougeLResults])),
        "Arabic_ROUGE_L_Recall": float(np.mean([item["recall"] for item in rougeLResults])),
        "Arabic_ROUGE_L_F1": float(np.mean([item["f1"] for item in rougeLResults])),

        "Arabic_ROUGE_S_Precision": float(np.mean([item["precision"] for item in rougeSResults])),
        "Arabic_ROUGE_S_Recall": float(np.mean([item["recall"] for item in rougeSResults])),
        "Arabic_ROUGE_S_F1": float(np.mean([item["f1"] for item in rougeSResults])),

        "Arabic_Token_F1": float(np.mean(tokenF1Scores)),
        "Arabic_Exact_Match": float(np.mean(exactMatchScores)),
        "No_Answer_Rate": float(np.mean(noAnswerScores)),
        "Average_Prediction_Length": float(np.mean(predictionLengths)),
        "Average_Reference_Length": float(np.mean(referenceLengths))
    }

    details = pd.DataFrame({
        "referenceAnswer": references,
        "prediction": predictions,
        "semanticSimilarity": semanticSimilarities.detach().cpu().numpy(),
        "arabicRougeL_F1": [item["f1"] for item in rougeLResults],
        "arabicRougeS_F1": [item["f1"] for item in rougeSResults],
        "arabicTokenF1": tokenF1Scores,
        "arabicExactMatch": exactMatchScores,
        "noAnswer": noAnswerScores,
        "predictionLength": predictionLengths,
        "referenceLength": referenceLengths
    })

    return metrics, details

def addColumnsSafely(targetDf, sourceDf, columns):
    for col in columns:
        if col in sourceDf.columns:
            if col in targetDf.columns:
                targetDf[col] = sourceDf[col].tolist()
            else:
                targetDf.insert(0, col, sourceDf[col].tolist())
    return targetDf

## 4. BM25-focused context functions

In [ ]:
def splitArabicSentences(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(r"(?<=[.!؟؛])\s+|(?<=،)\s+|\n+", text)
    sentences = [sentence.strip() for sentence in sentences if len(sentence.strip()) > 15]

    return sentences

def tokenizeForBm25(text):
    text = normalizeArabicText(text)
    return text.split()

def trimByWords(text, maxWords):
    words = str(text).split()

    if len(words) <= maxWords:
        return str(text).strip()

    return " ".join(words[:maxWords]).strip()

def buildBm25FocusedContext(contextText, questionText, topK=5, windowSize=1, maxWords=320):
    sentences = splitArabicSentences(contextText)

    if len(sentences) == 0:
        return trimByWords(contextText, maxWords)

    tokenizedSentences = [tokenizeForBm25(sentence) for sentence in sentences]
    queryTokens = tokenizeForBm25(questionText)

    if len(queryTokens) == 0:
        return trimByWords(" ".join(sentences[:topK]), maxWords)

    bm25 = BM25Okapi(tokenizedSentences)
    scores = bm25.get_scores(queryTokens)

    rankedIndexes = sorted(
        range(len(scores)),
        key=lambda index: scores[index],
        reverse=True
    )

    selectedIndexes = set()

    for index in rankedIndexes[:topK]:
        startIndex = max(0, index - windowSize)
        endIndex = min(len(sentences), index + windowSize + 1)

        for sentenceIndex in range(startIndex, endIndex):
            selectedIndexes.add(sentenceIndex)

    selectedIndexes = sorted(selectedIndexes)

    focusedSentences = []
    currentWordCount = 0

    for index in selectedIndexes:
        sentence = sentences[index]
        sentenceWords = sentence.split()

        if currentWordCount + len(sentenceWords) > maxWords:
            remainingWords = maxWords - currentWordCount

            if remainingWords > 20:
                focusedSentences.append(" ".join(sentenceWords[:remainingWords]))

            break

        focusedSentences.append(sentence)
        currentWordCount += len(sentenceWords)

    focusedContext = " ".join(focusedSentences).strip()

    if focusedContext == "":
        focusedContext = trimByWords(" ".join(sentences[:topK]), maxWords)

    return focusedContext

## 5. Load data and create the same leakage-aware split

In [ ]:
def contextHash(text):
    text = normalizeText(text)
    return hashlib.md5(text.encode("utf-8")).hexdigest()

df = pd.read_csv(dataPath).fillna("")

requiredColumns = ["instruction", "question", "context_text", "answer"]
for col in requiredColumns:
    if col not in df.columns:
        raise ValueError(f"missing required column: {col}")

if "ref_link" not in df.columns:
    df["ref_link"] = ""

df["rowId"] = np.arange(len(df))
df["ref_link"] = df["ref_link"].astype(str).str.strip()
df["context_text"] = df["context_text"].astype(str)
df["question"] = df["question"].astype(str)
df["answer"] = df["answer"].astype(str)
df["targetText"] = df["answer"].apply(cleanReference)

df["groupKey"] = df["ref_link"]
emptyGroupMask = df["groupKey"].eq("") | df["groupKey"].str.lower().isin(["nan", "none"])
df.loc[emptyGroupMask, "groupKey"] = df.loc[emptyGroupMask, "context_text"].apply(contextHash)

uniqueGroups = df["groupKey"].drop_duplicates().sample(frac=1.0, random_state=seed).tolist()

trainEnd = int(len(uniqueGroups) * trainRatio)
validationEnd = int(len(uniqueGroups) * (trainRatio + validationRatio))

trainGroups = set(uniqueGroups[:trainEnd])
validationGroups = set(uniqueGroups[trainEnd:validationEnd])
testGroups = set(uniqueGroups[validationEnd:])

trainDf = df[df["groupKey"].isin(trainGroups)].sample(frac=1.0, random_state=seed).reset_index(drop=True)
validationDf = df[df["groupKey"].isin(validationGroups)].sample(frac=1.0, random_state=seed).reset_index(drop=True)
testDf = df[df["groupKey"].isin(testGroups)].sample(frac=1.0, random_state=seed).reset_index(drop=True)

print("rows:", len(df))
print("train:", len(trainDf))
print("validation:", len(validationDf))
print("test:", len(testDf))
print("groups:", len(uniqueGroups))

rows: 17446
train: 13614
validation: 1978
test: 1854
groups: 3084


## 6. Build BM25-focused context for train, validation, and test

In [ ]:
def addFocusedContext(splitDf, splitName):
    splitDf = splitDf.copy()

    tqdm.pandas(desc=f"building BM25-focused contexts for {splitName}")

    splitDf["bm25FocusedContext"] = splitDf.progress_apply(
        lambda row: buildBm25FocusedContext(
            contextText=row["context_text"],
            questionText=row["question"],
            topK=focusedTopK,
            windowSize=focusedWindowSize,
            maxWords=focusedMaxWords
        ),
        axis=1
    )

    splitDf["bm25FocusedContextWords"] = splitDf["bm25FocusedContext"].apply(lambda text: len(str(text).split()))

    return splitDf

trainDf = addFocusedContext(trainDf, "train")
validationDf = addFocusedContext(validationDf, "validation")
testDf = addFocusedContext(testDf, "test")

focusedDataPreviewPath = resultsDir / "bm25_focused_data_preview.csv"
pd.concat([
    trainDf.head(5).assign(split="train"),
    validationDf.head(5).assign(split="validation"),
    testDf.head(5).assign(split="test")
]).to_csv(focusedDataPreviewPath, index=False, encoding="utf-8-sig")

print("saved preview:", focusedDataPreviewPath)
testDf[["question", "bm25FocusedContext", "bm25FocusedContextWords"]].head()

building BM25-focused contexts for train:   0%|          | 0/13614 [00:00<?, ?it/s]

building BM25-focused contexts for validation:   0%|          | 0/1978 [00:00<?, ?it/s]

building BM25-focused contexts for test:   0%|          | 0/1854 [00:00<?, ?it/s]

saved preview: outputs/full_context_bm25_full-tuning_results/bm25_focused_data_preview.csv


,question,bm25FocusedContext,bm25FocusedContextWords
0,ماذا ورد عن القدس في الجزء الثاني من شعارات ال...,"وتنتقل رسومه للصحف العربية. وأصبحت شخصية ""حنظل...",172
1,كيف يظهر دور الاحتلال أو السلطات الإسرائيلية ف...,نبش مقام أبو الهيجاء حليف صلاح الدين الأيوبي و...,167
2,ما الأماكن أو المواقع المرتبطة بـالجزء الأول م...,شعارات العودة والمواقف السياسية جدران المخيم.....,123
3,أين استقر معظم آل موسى في أوائل فترة الانتداب ...,آل موسى وهم من التلاوية / بكاروه / يقال الأصل ...,103
4,كيف حوصرت قرية دير ياسين عند بدء المجزرة؟,متى حصلت المجزرة؟ وقعت المجزرة فجر 9 أبريل/نيس...,151


## 7. Build model inputs

In [ ]:
def buildBm25FocusedInput(row):
    instruction = str(row.get("instruction", "")).strip()

    if instruction == "":
        instruction = "أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، متخصص في روايات القرى والمدن الفلسطينيه. أجب بناءً على الشهادات الشفوية والمصادر المحلية الموثّقة، مع الحفاظ على أصالة السياق الإنساني."

    return f"""أجب عن السؤال التالي باللغة العربية اعتمادًا على السياق المرفق فقط.
إذا لم تكن الإجابة موجودة في السياق، قل: لا تتوفر معلومات كافية.

التعليمة:
{instruction}

السؤال:
{row["question"]}

السياق:
{row["bm25FocusedContext"]}"""

trainDf["inputText"] = trainDf.apply(buildBm25FocusedInput, axis=1)
validationDf["inputText"] = validationDf.apply(buildBm25FocusedInput, axis=1)
testDf["inputText"] = testDf.apply(buildBm25FocusedInput, axis=1)

trainRawDataset = Dataset.from_pandas(trainDf[["inputText", "targetText"]], preserve_index=False)
validationRawDataset = Dataset.from_pandas(validationDf[["inputText", "targetText"]], preserve_index=False)
testRawDataset = Dataset.from_pandas(testDf[["inputText", "targetText"]], preserve_index=False)

print(trainDf["inputText"].iloc[0][:1000])

أجب عن السؤال التالي باللغة العربية اعتمادًا على السياق المرفق فقط.
إذا لم تكن الإجابة موجودة في السياق، قل: لا تتوفر معلومات كافية.

التعليمة:
أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، متخصص في روايات القرى والمدن الفلسطينيه. أجب بناءً على الشهادات الشفوية والمصادر المحلية الموثّقة، مع الحفاظ على أصالة السياق الإنساني.

السؤال:
ماذا ورد عن الجزء الأول من اللاجئون في الوطن.. صورٌ لا تُمحى ونكبةٌ مستمرة!؟

السياق:
اللاجئون في الوطن.. صورٌ لا تُمحى ونكبةٌ مستمرة! في عمره وهو الذي تجاوز الـ 83، لا زالَ الحاج أبو فهيم، عبد الرحمن كيال، وهي أيضًا مولودة في البروة. عن بروته يقول: "كان عدد سكان البروة في العام 1948، وكان قد تزوّج أشقائي قبل النكبة، ليدخل جيش الإنقاذ ويهاجم البلدة ويحررها، أما عائلتي فقد خرجنا في الهجوم الأول إذ أحاطتنا ثلاث مصفحات، تركوا لنا الطريق شرقًا كي نخرج،


## 8. Tokenizer and preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(modelStartPath)

def preprocessExamples(examples):
    modelInputs = tokenizer(
        examples["inputText"],
        max_length=maxInputLength,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["targetText"],
        max_length=maxTargetLength,
        truncation=True
    )

    modelInputs["labels"] = labels["input_ids"]

    return modelInputs

trainDataset = trainRawDataset.map(
    preprocessExamples,
    batched=True,
    remove_columns=trainRawDataset.column_names
)

validationDataset = validationRawDataset.map(
    preprocessExamples,
    batched=True,
    remove_columns=validationRawDataset.column_names
)

dataCollator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print(trainDataset)
print(validationDataset)

/home/rubaegb/my_venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/rubaegb/my_venv/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/13614 [00:00<?, ? examples/s]

Map:   0%|          | 0/1978 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 13614
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1978
})


## 9. Load model and train on BM25-focused context

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(modelStartPath)
model.to(device)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    useBf16 = major >= 8
    useFp16 = not useBf16

trainingArgs = Seq2SeqTrainingArguments(
    output_dir=str(modelOutputDir),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learningRate,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=gradientAccumulationSteps,
    weight_decay=weightDecay,
    num_train_epochs=numTrainEpochs,
    warmup_ratio=warmupRatio,
    lr_scheduler_type="constant_with_warmup",
    generation_max_length=maxTargetLength,
    predict_with_generate=True,
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=False,
    bf16=torch.cuda.is_available(),
    seed=seed,
    data_seed=seed,
    save_safetensors=False,
    gradient_checkpointing=False if freezeEncoder else True,
    max_grad_norm=1.0,


)

trainer = Seq2SeqTrainer(
    model=model,
    args=trainingArgs,
    train_dataset=trainDataset,
    eval_dataset=validationDataset,
    tokenizer=tokenizer,
    data_collator=dataCollator
)
from pathlib import Path

checkpoints = sorted(
    modelOutputDir.glob("checkpoint-*"),
    key=lambda path: path.stat().st_mtime,
    reverse=True
)

resumeCheckpoint = str(checkpoints[0]) if len(checkpoints) > 0 else None

trainOutput = trainer.train(resume_from_checkpoint=resumeCheckpoint)
trainer.save_model(str(modelOutputDir))
tokenizer.save_pretrained(str(modelOutputDir))

print("saved trained BM25-focused model to:", modelOutputDir)
print(trainOutput)

/home/rubaegb/my_venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
0,1.146600,1.041078
1,1.111000,0.950515
2,0.970400,0.929535
3,0.948600,0.919957
4,0.813300,0.912864


/home/rubaegb/my_venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/rubaegb/my_venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/rubaegb/my_venv/lib/python

saved trained BM25-focused model to: outputs/google/mt5-large-bm25-full-tuning
TrainOutput(global_step=4250, training_loss=1.808122776704676, metrics={'train_runtime': 16045.3949, 'train_samples_per_second': 4.242, 'train_steps_per_second': 0.265, 'total_flos': 2.079440271998976e+17, 'train_loss': 1.808122776704676, 'epoch': 4.994858234170707})


In [ ]:
from pathlib import Path
import os
import gc
import torch
import random
import numpy as np

step = trainer.state.global_step
checkpointDir = Path(modelOutputDir) / f"checkpoint-{step}-manual-rescue"
checkpointDir.mkdir(parents=True, exist_ok=True)

print("Saving rescue checkpoint to:", checkpointDir)
print("Current trainer step:", step)
print("Current epoch:", trainer.state.epoch)

trainer.model.save_pretrained(
    str(checkpointDir),
    safe_serialization=False
)

tokenizer.save_pretrained(str(checkpointDir))

trainer.state.save_to_json(str(checkpointDir / "trainer_state.json"))

torch.save(trainingArgs, checkpointDir / "training_args.bin")

if trainer.optimizer is not None:
    torch.save(trainer.optimizer.state_dict(), checkpointDir / "optimizer.pt")
    print("Saved optimizer.pt")

if trainer.lr_scheduler is not None:
    torch.save(trainer.lr_scheduler.state_dict(), checkpointDir / "scheduler.pt")
    print("Saved scheduler.pt")

rngState = {
    "python": random.getstate(),
    "numpy": np.random.get_state(),
    "cpu": torch.random.get_rng_state(),
}

if torch.cuda.is_available():
    rngState["cuda"] = torch.cuda.random.get_rng_state_all()

torch.save(rngState, checkpointDir / "rng_state.pth")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("DONE. Manual rescue checkpoint saved.")
print("Checkpoint path:", checkpointDir)

Saving rescue checkpoint to: outputs/google/mt5-large-bm25-full-tuning/checkpoint-4250-manual-rescue
Current trainer step: 4250
Current epoch: 4.994858234170707
Saved optimizer.pt
Saved scheduler.pt
DONE. Manual rescue checkpoint saved.
Checkpoint path: outputs/google/mt5-large-bm25-full-tuning/checkpoint-4250-manual-rescue


In [ ]:
import os
print(os.listdir(checkpointDir))

['tokenizer_config.json', 'trainer_state.json', 'tokenizer.json', 'training_args.bin', 'optimizer.pt', 'config.json', 'scheduler.pt', 'rng_state.pth', 'spiece.model', 'pytorch_model.bin', 'special_tokens_map.json', 'generation_config.json']


## 10. Save training log

In [ ]:
logHistoryDf = pd.DataFrame(trainer.state.log_history)
trainingLogPath = resultsDir / "bm25_focused_training_log.csv"
logHistoryDf.to_csv(trainingLogPath, index=False, encoding="utf-8-sig")

logHistoryDf.tail()

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
171,0.8773,1.213581,0.00005,4.936095,4200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
172,0.9669,0.868935,0.00005,4.965477,4225,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
173,0.8133,1.174923,0.00005,4.994858,4250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
174,NaN,NaN,NaN,4.994858,4250,0.912864,93.2043,21.222,10.611,NaN,NaN,NaN,NaN,NaN
175,NaN,NaN,NaN,4.994858,4250,NaN,NaN,NaN,NaN,16045.3949,4.242,0.265,2.079440e+17,1.808123


## 11. Generate predictions from the trained BM25-focused model

In [ ]:
def getBadWordsIds(tokenizer):
    badTokens = [f"<extra_id_{index}>" for index in range(100)]
    badIds = []

    for token in badTokens:
        tokenIds = tokenizer.encode(token, add_special_tokens=False)

        if len(tokenIds) > 0:
            badIds.append(tokenIds)

    return badIds

def generateAnswersCheckpointed(model, tokenizer, inputTexts, outputCsvPath, batchSize=4, maxNewTokens=256, numBeams=1, forceRegenerate=False):
    outputCsvPath = Path(outputCsvPath)

    if outputCsvPath.exists() and not forceRegenerate:
        existingDf = pd.read_csv(outputCsvPath).fillna("")
        existingDf = existingDf.sort_values("rowIndex").drop_duplicates("rowIndex")
# in case we want to get the cached ones, so we don't redo it 
# which is what we want, but in this specific run i commented it because i mistakly did not load the model so i want it to recalculate
#         if len(existingDf) == len(inputTexts):
#             print("loading cached predictions:", outputCsvPath)
#             return existingDf["prediction"].tolist()

#         print("cached predictions are incomplete, regenerating from scratch.")
#         outputCsvPath.unlink()

    predictions = []
    badWordsIds = getBadWordsIds(tokenizer)

    for startIndex in tqdm(range(0, len(inputTexts), batchSize)):
        batchTexts = inputTexts[startIndex:startIndex + batchSize]

        encoded = tokenizer(
            batchTexts,
            max_length=maxInputLength,
            truncation=True,
            padding=True,
            return_tensors="pt"
        )

        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            generatedIds = model.generate(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
                max_new_tokens=maxNewTokens,
                num_beams=numBeams,
                bad_words_ids=badWordsIds,
                early_stopping=True
            )

        batchPredictions = tokenizer.batch_decode(generatedIds, skip_special_tokens=True)
        predictions.extend(batchPredictions)

    pd.DataFrame({
        "rowIndex": list(range(len(predictions))),
        "prediction": predictions
    }).to_csv(outputCsvPath, index=False, encoding="utf-8-sig")

    return predictions

testInputs = testDf["inputText"].tolist()
references = testDf["targetText"].tolist()

predictionsPath = resultsDir / "bm25_focused_trained_test_predictions.csv"

bm25FocusedPredictions = generateAnswersCheckpointed(
    model=model,
    tokenizer=tokenizer,
    inputTexts=testInputs,
    outputCsvPath=predictionsPath,
    batchSize=evalBatchSize,
    maxNewTokens=generationMaxNewTokens,
    numBeams=generationNumBeams,
    forceRegenerate=forceRegeneratePredictions
)

cleanedBm25FocusedPredictions = [
    cleanGeneratedText(prediction)
    for prediction in bm25FocusedPredictions
]

print("predictions:", len(cleanedBm25FocusedPredictions))
print(cleanedBm25FocusedPredictions[:3])

  0%|          | 0/232 [00:00<?, ?it/s]

/home/rubaegb/my_venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:615: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


predictions: 1854
['وقال قرعان إن معظم تجارب الرسم هذه تخرج بصورة تجميعية تبرز حقوق اللاجئين وسواد المخيم وواقعه المؤلم المحاط بأشجار الصبر عادة، دون أن يهمل المشاركون بالرسم أسوار القدس والصعود إليها لتحريرها.', 'وحاول المعتدون نبش مقام الشيخ المجاور لقرية كوكب أبو الهيجاء في قضاء الناصرة والعبث به، وأقدموا على تنفيذ الهجوم في جنح الظلام دون أن يمسوا بصندوق التبرعات المثبت في المكان، وقاتل الصليبيين في فلسطين ودفن فيها بعدما أسس عدة قرى عرفت باسمه.', 'ترتبط القضية بـقلنديا، القدس المحتلة. قبيل أيام من ذكرى نكبة فلسطين كل عام ينشط الشاب سعيد أصلان ورفاقه في مخيم قلنديا للاجئين الفلسطينيين شمال القدس المحتلة، بنشر الشعارات الوطنية ورسم تطلعاتهم للعودة إلى قراهم المهجرة عام 1948 في لوحات على جدران المخيم.']


## 12. Compute corrected metrics

In [ ]:
bm25FocusedMetrics, bm25FocusedDetailsDf = computeMetricsHonest(
    cleanedBm25FocusedPredictions,
    references
)

bm25FocusedMetrics["system"] = "fine_tuned_mt5_large_bm25_focused_trained"
bm25FocusedMetrics["contextType"] = "bm25_focused_gold_context"
bm25FocusedMetrics["Max_Input_Tokens"] = maxInputLength
bm25FocusedMetrics["Max_New_Tokens"] = maxTargetLength
bm25FocusedMetrics["Focused_TopK"] = focusedTopK
bm25FocusedMetrics["Focused_Window_Size"] = focusedWindowSize
bm25FocusedMetrics["Focused_Max_Words"] = focusedMaxWords
bm25FocusedMetrics["Trained_From"] = str(modelStartPath)

bm25FocusedMetricsDf = pd.DataFrame([bm25FocusedMetrics])

extraDetailsDf = pd.DataFrame({
    "rowId": testDf["rowId"].tolist(),
    "question": testDf["question"].tolist(),
    "referenceAnswer": references,
    "bm25FocusedPrediction": bm25FocusedPredictions,
    "cleanedBm25FocusedPrediction": cleanedBm25FocusedPredictions,
    "bm25FocusedContext": testDf["bm25FocusedContext"].tolist(),
    "bm25FocusedContextWords": testDf["bm25FocusedContextWords"].tolist(),
    "inputText": testInputs
})

bm25FocusedDetailsDf = addColumnsSafely(
    bm25FocusedDetailsDf,
    extraDetailsDf,
    [
        "rowId",
        "question",
        "referenceAnswer",
        "bm25FocusedPrediction",
        "cleanedBm25FocusedPrediction",
        "bm25FocusedContext",
        "bm25FocusedContextWords",
        "inputText"
    ]
)

metricsPath = resultsDir / "bm25_focused_trained_test_metrics.csv"
detailsPath = resultsDir / "bm25_focused_trained_test_prediction_details.csv"

bm25FocusedMetricsDf.to_csv(metricsPath, index=False, encoding="utf-8-sig")
bm25FocusedDetailsDf.to_csv(detailsPath, index=False, encoding="utf-8-sig")

print("saved:", metricsPath)
print("saved:", detailsPath)
print(bm25FocusedMetricsDf.T)

/home/rubaegb/my_venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

Batches:   0%|          | 0/58 [00:00<?, ?it/s]

saved: outputs/full_context_bm25_full-tuning_results/bm25_focused_trained_test_metrics.csv
saved: outputs/full_context_bm25_full-tuning_results/bm25_focused_trained_test_prediction_details.csv
                                                                     0
BLEU                                                         38.555018
chrF                                                         51.133056
BERTScore_Precision                                           0.830846
BERTScore_Recall                                              0.825344
BERTScore_F1                                                  0.826912
Semantic_Textual_Similarity                                   0.730777
Arabic_ROUGE_L_Precision                                      0.507839
Arabic_ROUGE_L_Recall                                         0.494998
Arabic_ROUGE_L_F1                                             0.479429
Arabic_ROUGE_S_Precision                                      0.386658
Arabic_ROUGE_S_Recall     

## 13. Create sample review

In [ ]:
sampleCount = min(10, len(bm25FocusedDetailsDf))

sampleReviewDf = bm25FocusedDetailsDf[[
    "question",
    "referenceAnswer",
    "cleanedBm25FocusedPrediction",
    "semanticSimilarity",
    "arabicRougeL_F1",
    "arabicRougeS_F1",
    "arabicTokenF1",
    "arabicExactMatch",
    "bm25FocusedContext"
]].head(sampleCount)

samplePath = resultsDir / "bm25_focused_trained_sample_prediction_review.csv"
sampleReviewDf.to_csv(samplePath, index=False, encoding="utf-8-sig")

print("saved:", samplePath)
sampleReviewDf

saved: outputs/full_context_bm25_full-tuning_results/bm25_focused_trained_sample_prediction_review.csv


,question,referenceAnswer,cleanedBm25FocusedPrediction,semanticSimilarity,arabicRougeL_F1,arabicRougeS_F1,arabicTokenF1,arabicExactMatch,bm25FocusedContext
0,ماذا ورد عن القدس في الجزء الثاني من شعارات ال...,وفي مهرجان للرسم، انكبت الفتيات غصون مشرف اللا...,وقال قرعان إن معظم تجارب الرسم هذه تخرج بصورة ...,0.810252,0.659574,0.637813,0.659574,0,"وتنتقل رسومه للصحف العربية. وأصبحت شخصية ""حنظل..."
1,كيف يظهر دور الاحتلال أو السلطات الإسرائيلية ف...,تواصل مسلسل الاعتداءات على المواقع الإسلامية ف...,وحاول المعتدون نبش مقام الشيخ المجاور لقرية كو...,0.632317,0.069767,0.010076,0.093023,0,نبش مقام أبو الهيجاء حليف صلاح الدين الأيوبي و...
2,ما الأماكن أو المواقع المرتبطة بـالجزء الأول م...,ارتبطت المادة بالأماكن التالية: القدس، قلنديا،...,ترتبط القضية بـقلنديا، القدس المحتلة. قبيل أيا...,0.887923,0.745098,0.740741,0.745098,0,شعارات العودة والمواقف السياسية جدران المخيم.....
3,أين استقر معظم آل موسى في أوائل فترة الانتداب ...,استقر معظم آل موسى في أوائل فترة الانتداب البر...,استقر معظم آل موسى في أوائل فترة الانتداب البر...,1.000000,1.000000,1.000000,1.000000,1,آل موسى وهم من التلاوية / بكاروه / يقال الأصل ...
4,كيف حوصرت قرية دير ياسين عند بدء المجزرة؟,فوجئ أهل دير ياسين بأصوات مكبرات تدعوهم إلى إخ...,كانت دير ياسين تحت تهديد القتل، وكانت هناك قنا...,0.438938,0.133333,0.016667,0.133333,0,متى حصلت المجزرة؟ وقعت المجزرة فجر 9 أبريل/نيس...
5,ما مسار نزوح عائلة أبو هنطش قبل استقرارها في ق...,تذكر الرواية أن خمسة إخوة نزحوا من جبل الدروز ...,نزح خمسة إخوة من جبل الدروز، ثم نزلوا بلاد عجل...,0.856080,0.500000,0.174672,0.500000,0,ففي تاريخ شرق الاردن وقبائلها لبك باشا ( ص 338...
6,كيف استُخدمت مجزرة دير ياسين في الحرب النفسية ...,استُغلت المجزرة في تهديد الفلسطينيين بمصير مشا...,استُخدمت مجزرة دير ياسين في نشر أخبارها ونشر أ...,0.369706,0.074074,0.000000,0.074074,0,"وأضاف أن ""لهذه العملية نتائج كبيرة غير متوقعة،..."
7,ما أبرز ما حدث في مبادرة لرفع علم فلسطين على ك...,انطلقت في الرابع والعشرين من الشهر الماضي حملة...,انطلقت في الرابع والعشرين من الشهر الماضي حملة...,0.955595,0.655172,0.628571,0.655172,0,انطلقت في الرابع والعشرين من الشهر الماضي حملة...
8,بأي مكان ارتبطت عدس في السجل الفلسطيني؟,ارتبطت عدس بـعنبتا.,ارتبطت عدس بـالقدس.,0.367612,0.666667,0.333333,0.666667,0,الدكتور عبد الرحمن عدس:أستاذ متخصص في التربية،...
9,كيف تحاول الجماعات الاستيطانية تهويد بلدة سلوان؟,المستوطنون والمسؤولون الإسرائيليون يشتركون في ...,تحاول الجماعات الاستيلاء على منازلها وتنفيذ أع...,0.773445,0.387097,0.321429,0.387097,0,ترفرف أعلام للاحتلال الإسرائيلي على منازل فلسط...


## 14. compare with corrected full-context results if available

In [ ]:
comparisonRows = []

correctedFullContextComparisonPath = Path(
    "./outputs/full_context_results_metric_only_corrected/baseline_vs_fine_tuned_test_metrics_corrected.csv"
)

if correctedFullContextComparisonPath.exists():
    correctedFullContextDf = pd.read_csv(correctedFullContextComparisonPath).fillna("")

    for _, row in correctedFullContextDf.iterrows():
        comparisonRows.append(row.to_dict())
else:
    print("corrected full-context comparison file not found:", correctedFullContextComparisonPath)

comparisonRows.append(bm25FocusedMetrics)

comparisonDf = pd.DataFrame(comparisonRows)

metricsForImprovement = [
    "BLEU",
    "chrF",
    "BERTScore_F1",
    "Semantic_Textual_Similarity",
    "Arabic_ROUGE_L_F1",
    "Arabic_ROUGE_S_F1",
    "Arabic_Token_F1",
    "Arabic_Exact_Match"
]

if "system" in comparisonDf.columns and "baseline_pretrained_mt5_large" in comparisonDf["system"].astype(str).tolist():
    baselineRow = comparisonDf[
        comparisonDf["system"].astype(str) == "baseline_pretrained_mt5_large"
    ].iloc[0]

    for metric in metricsForImprovement:
        if metric in comparisonDf.columns:
            comparisonDf[f"{metric}_improvement_vs_baseline"] = (
                comparisonDf[metric].astype(float) - float(baselineRow[metric])
            )

goldMask = (
    (comparisonDf.get("system", pd.Series(dtype=str)).astype(str) == "fine_tuned_mt5_large")
    & (comparisonDf.get("contextType", pd.Series(dtype=str)).astype(str) == "gold_full_context")
)

if goldMask.any():
    fineTunedGoldRow = comparisonDf[goldMask].iloc[0]

    for metric in metricsForImprovement:
        if metric in comparisonDf.columns:
            comparisonDf[f"{metric}_delta_vs_fine_tuned_gold_context"] = (
                comparisonDf[metric].astype(float) - float(fineTunedGoldRow[metric])
            )

comparisonPath = resultsDir / "corrected_full_context_vs_bm25_focused_trained_metrics.csv"
comparisonDf.to_csv(comparisonPath, index=False, encoding="utf-8-sig")

print("saved:", comparisonPath)
comparisonDf

saved: outputs/full_context_bm25_full-tuning_results/corrected_full_context_vs_bm25_focused_trained_metrics.csv


,system,contextType,Max_Input_Tokens,Max_New_Tokens,BLEU,chrF,BERTScore_F1,Semantic_Textual_Similarity,Arabic_ROUGE_L_F1,Arabic_ROUGE_S_F1,...,Focused_Max_Words,Trained_From,BLEU_delta_vs_fine_tuned_gold_context,chrF_delta_vs_fine_tuned_gold_context,BERTScore_F1_delta_vs_fine_tuned_gold_context,Semantic_Textual_Similarity_delta_vs_fine_tuned_gold_context,Arabic_ROUGE_L_F1_delta_vs_fine_tuned_gold_context,Arabic_ROUGE_S_F1_delta_vs_fine_tuned_gold_context,Arabic_Token_F1_delta_vs_fine_tuned_gold_context,Arabic_Exact_Match_delta_vs_fine_tuned_gold_context
0,baseline_pretrained_mt5_large,gold_full_context,1024,256,0.565116,16.835745,0.604758,0.240820,0.052862,0.008943,...,NaN,NaN,-12.227775,-11.420246,-0.120684,-0.269258,-0.152387,-0.099573,-0.162808,-0.003429
1,fine_tuned_mt5_large,gold_full_context,1024,256,12.792891,28.255991,0.725442,0.510078,0.205249,0.108516,...,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,fine_tuned_mt5_large_bm25_focused_trained,bm25_focused_gold_context,1024,256,38.555018,51.133056,0.826912,0.730777,0.479429,0.358925,...,320.0,google/mt5-large,25.762127,22.877065,0.101470,0.220698,0.274180,0.250408,0.270592,0.096895
